# Klasifikasi KNN Manual (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini mengimplementasikan **K-Nearest Neighbors (KNN)** dari nol, tanpa
scikit-learn. Model dan metriknya dihitung sendiri. Datanya dibaca dari berkas bersih
hasil notebook `02_bersihkan_data`, jadi tidak ada library machine learning yang dipakai.

Tujuannya bukan mengejar skor tertinggi, melainkan melihat cara kerja algoritma dari
dalam: menghitung jarak, mengambil tetangga terdekat, lalu voting.

Dataset yang dipakai: Stroke Prediction versi bersih dari notebook `02_bersihkan_data`
(5.109 baris). Hanya kolom numerik yang dipakai: age, hypertension, heart_disease,
avg_glucose_level, bmi. Target: stroke.

Pasangan notebook ini untuk Naive Bayes ada di `03b_naive_bayes_manual.ipynb`.
Versi scikit-learn menyusul di `04_model_library.ipynb`.


## 1. Muat Dataset

Data diambil langsung dari **berkas dataset bersih** yang dihasilkan notebook
`02_bersihkan_data`, yaitu `artifacts/stroke_bersih.csv` (5.109 baris, `bmi` sudah diisi
prediksi Ridge, `gender` "Other" sudah dibuang).

Karena pembersihannya sudah selesai di notebook `02`, notebook ini bisa fokus penuh ke
algoritmanya. Kolom `gender` tetap dimuat karena dibutuhkan skor medis di bagian 5.

In [ ]:
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

# Dataset bersih hasil notebook 02_bersihkan_data, diambil dari repositori supaya
# isinya sama persis dengan yang dipakai notebook lain.
URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")

KOLOM = ["gender", "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]
df = pd.read_csv(URL_BERSIH)[KOLOM]

# Fitur untuk model. gender tidak dipakai sebagai fitur, hanya untuk skor medis di bagian 5.
fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

print("Ukuran dataset bersih:", df.shape)
print("BMI kosong:", df["bmi"].isna().sum())
print("5 baris pertama:")
df.head()

## 2. Train-Test Split Manual

Split 80:20 tanpa library. Acak indeks dulu, lalu potong. Karena datanya tidak
seimbang (hanya 4,87% stroke), proporsi kelas di train dan test dicetak supaya
kelihatan apakah kedua bagian kebagian kasus stroke.

In [ ]:
def train_test_split_manual(data, test_size=0.2, seed=42):
    """Bagi baris data jadi train dan test secara manual (80:20)."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return data.iloc[train_idx], data.iloc[test_idx]


latih, uji = train_test_split_manual(df, test_size=0.2, seed=42)

X_train, y_train = latih[fitur].values, latih["stroke"].values
X_test, y_test = uji[fitur].values, uji["stroke"].values

print(f"Train: {len(latih)} baris")
print(f"Test:  {len(uji)} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

## 3. Fungsi Evaluasi Manual

Semua metrik dihitung dari confusion matrix tanpa library.

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 4. KNN Manual

Hitung jarak Euclidean ke semua data train, ambil K tetangga terdekat, lalu voting
mayoritas.

In [ ]:
def euclidean_distance(a, b):
    """Jarak Euclidean antara dua vektor."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


def knn_predict(X_train, y_train, X_test, k=5):
    """KNN: prediksi kelas untuk tiap titik uji."""
    y_pred = []

    for titik_uji in X_test:
        jarak = []
        for i, titik_latih in enumerate(X_train):
            d = euclidean_distance(titik_uji, titik_latih)
            jarak.append((d, y_train[i]))

        jarak.sort(key=lambda x: x[0])
        tetangga = jarak[:k]

        voting = 0
        for _, label in tetangga:
            voting += label

        pred = 1 if voting > k / 2 else 0
        y_pred.append(pred)

    return np.array(y_pred)


print("Menjalankan KNN (k=5)...")
y_pred_knn = knn_predict(X_train, y_train, X_test, k=5)
print("Selesai.")

In [ ]:
metrik_knn = hitung_metrik(y_test, y_pred_knn)

print("=" * 50)
print("HASIL KNN (k=5)")
print("=" * 50)
print(f"  Akurasi:  {metrik_knn['akurasi']}")
print(f"  Presisi:  {metrik_knn['presisi']}")
print(f"  Recall:   {metrik_knn['recall']}")
print(f"  F1-score: {metrik_knn['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_knn['tp']}  FP={metrik_knn['fp']}")
print(f"  FN={metrik_knn['fn']}  TN={metrik_knn['tn']}")

### 4a. Coba Beberapa Nilai K

Cari nilai K yang paling optimal dengan mencoba beberapa angka. K kecil peka derau,
K besar cenderung menebak kelas mayoritas.

In [ ]:
print(f"{'K':>3} {'Akurasi':>8} {'Presisi':>8} {'Recall':>8} {'F1':>8}")
print("-" * 40)

for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    pred = knn_predict(X_train, y_train, X_test, k=k)
    m = hitung_metrik(y_test, pred)
    print(f"{k:>3} {m['akurasi']:>8} {m['presisi']:>8} {m['recall']:>8} {m['f1']:>8}")

## 5. Pembanding: Skor Risiko Stroke Klinis (CHA2DS2-VASc)

Sebelum ada machine learning, dunia medis sudah punya cara menghitung risiko stroke.
Salah satu yang paling dikenal adalah **CHA2DS2-VASc**, skor yang dipakai untuk
memperkirakan risiko stroke pada pasien fibrilasi atrium.

Skornya hanyalah penjumlahan poin:

$$\text{Skor} = C + H + A_2 + D + S_2 + V + A + Sc$$

dengan tiap komponen bernilai:

| Lambang | Arti | Poin |
|---|---|---|
| $C$ | Congestive heart failure (gagal jantung) | 1 |
| $H$ | Hypertension (hipertensi) | 1 |
| $A_2$ | Age $\geq$ 75 tahun | 2 |
| $D$ | Diabetes mellitus | 1 |
| $S_2$ | Stroke atau TIA sebelumnya | 2 |
| $V$ | Vascular disease (penyakit vaskular) | 1 |
| $A$ | Age 65 sampai 74 tahun | 1 |
| $Sc$ | Sex category (perempuan) | 1 |

### Adaptasi ke dataset kami

Dua komponen tidak bisa dipakai, dan ini harus disebut apa adanya:

- $S_2$ (riwayat stroke) justru **label yang mau diprediksi**, jadi tidak boleh jadi fitur.
- $V$ (penyakit vaskular) tidak ada di dataset.

Sisanya dipakai apa adanya, dengan dua proksi yang juga harus disebut:

- $C$ memakai kolom `heart_disease`
- $D$ memakai `avg_glucose_level >= 126` mg/dL, ambang diabetes menurut ADA

Karena itu skor maksimum di sini **6**, bukan 9.

Skor ini tidak dilatih dan tidak belajar dari data, ia aturan tetap. Justru itu gunanya
di sini: jadi pembanding "cara dokter" terhadap "cara machine learning".

In [ ]:
def skor_cha2ds2_vasc(baris):
    """CHA2DS2-VASc versi adaptasi. Penjelasan rumusnya ada di sel atas."""
    skor = 0
    if baris["heart_disease"] == 1:          # C, proksi gagal jantung
        skor += 1
    if baris["hypertension"] == 1:           # H
        skor += 1
    if baris["age"] >= 75:                   # A2
        skor += 2
    elif baris["age"] >= 65:                 # A
        skor += 1
    if baris["avg_glucose_level"] >= 126:    # D, ambang diabetes menurut ADA
        skor += 1
    if baris["gender"] == "Female":          # Sc
        skor += 1
    # S2 (riwayat stroke) sengaja tidak dipakai: itu label yang sedang diprediksi.
    # V (penyakit vaskular) tidak tersedia di dataset.
    return skor


skor = df.apply(skor_cha2ds2_vasc, axis=1)
skor_uji = skor.loc[uji.index].values

print("Sebaran skor pada data uji:")
print(pd.Series(skor_uji).value_counts().sort_index().to_string())

In [ ]:
baris_skor = []
for ambang in [1, 2, 3, 4]:
    pred = (skor_uji >= ambang).astype(int)
    m = hitung_metrik(y_test, pred)
    baris_skor.append({
        "Aturan": f"skor >= {ambang}",
        "akurasi": m["akurasi"],
        "presisi": m["presisi"],
        "recall": m["recall"],
        "f1": m["f1"],
        "ditandai": int(pred.sum()),
    })

pd.DataFrame(baris_skor)

## 6. Uji Kombinasi Atribut terhadap Confidence Level

Arahan dosen pada Pertemuan 3 meminta menentukan **fitur utama** dan **kombinasi atribut
terbaik untuk confidence level, bukan akurasi**. Notebook `13` sudah menjawabnya memakai
Logistic Regression. Di sini hal yang sama diulang memakai **KNN manual**:
delapan kombinasi atribut diadu, lalu diukur confidence level-nya.

**Kenapa precision, bukan akurasi.** Hanya 4,87% pasien yang benar-benar stroke, jadi
akurasi menyesatkan (bagian sebelumnya sudah memperlihatkannya). Confidence di sini
diukur dengan **precision**, dan satu angka ringkasannya **average precision (AP)**,
yaitu luas di bawah kurva precision-recall. Dicatat juga precision pada recall 0,80,
titik kerja yang dipakai notebook `07`.

**Protokol yang adil.** Semua kombinasi diuji pada baris dan lipatan yang persis sama:
lima lipatan stratified, skor diambil out-of-fold (tiap pasien dinilai oleh model yang
tidak pernah melihatnya saat latih). Setiap AP diberi selang kepercayaan bootstrap 95%.
Protokolnya meniru notebook `13`; lipatannya dibangun manual, jadi angkanya boleh
berbeda tipis dari jalur scikit-learn di sana.


### 6a. Menyusun delapan kombinasi atribut

Mesin uji ini memakai seluruh kolom data bersih, bukan hanya lima kolom numerik di bagian
awal, karena kombinasinya mencakup atribut sosial (pekerjaan, wilayah, status merokok).
Kolom kategorikal diubah menjadi one-hot.


In [ ]:
df_penuh = pd.read_csv(URL_BERSIH)

y_komb = df_penuh["stroke"].astype(int).to_numpy()
X_dasar = pd.get_dummies(df_penuh.drop(columns=["stroke"]), drop_first=True).astype(float)

print("Ukuran data bersih   :", df_penuh.shape)
print(f"Kasus stroke         : {int(y_komb.sum())} ({y_komb.mean()*100:.2f}%)")
print(f"Fitur dasar (one-hot): {X_dasar.shape[1]}")


Fitur turunan if-else dari laporan Pertemuan 3 ikut diuji supaya aturannya terukur,
bukan hanya diklaim. Nilainya ordinal, makin besar makin berisiko.


In [ ]:
def fitur_turunan(d):
    """Fitur turunan if-else, disalin dari laporan Pertemuan 3. Nilainya ordinal:
    semakin besar semakin berisiko."""
    f = pd.DataFrame(index=d.index)
    f["glukosa_kat"] = np.digitize(d["avg_glucose_level"], [100, 140, 200]).astype(float)
    f["bmi_kat"] = np.digitize(d["bmi"], [18.5, 25, 30]).astype(float)
    f["jumlah_faktor"] = (d["hypertension"] + d["heart_disease"]
                          + (d["avg_glucose_level"] > 200).astype(int)
                          + (d["bmi"] >= 30).astype(int)
                          + (d["age"] >= 60).astype(int)).astype(float)
    f["risiko_nested"] = 0.0
    f.loc[(d.hypertension == 1) | (d.heart_disease == 1), "risiko_nested"] = 1
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1), "risiko_nested"] = 2
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1)
          & (d.avg_glucose_level > 200), "risiko_nested"] = 3
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1)
          & (d.avg_glucose_level > 200) & (d.age > 60), "risiko_nested"] = 4
    return f


TURUNAN = ["glukosa_kat", "bmi_kat", "jumlah_faktor", "risiko_nested"]
X_komb = pd.concat([X_dasar, fitur_turunan(df_penuh)], axis=1).astype(float)

print("Fitur dasar   :", X_dasar.shape[1])
print("Fitur turunan :", len(TURUNAN))
print("Total fitur   :", X_komb.shape[1])


Kelompok atribut dipetakan ke kolomnya. Sebuah kelompok kategorikal dicocokkan ke
kolom one-hot-nya, misalnya `gender` cocok dengan `gender_Male`.


In [ ]:
GRUP = {
    "usia (age)": ["age"],
    "hipertensi": ["hypertension"],
    "penyakit jantung": ["heart_disease"],
    "glukosa": ["avg_glucose_level"],
    "bmi": ["bmi"],
    "jenis kelamin": ["gender"],
    "status menikah": ["ever_married"],
    "pekerjaan": ["work_type"],
    "wilayah": ["Residence_type"],
    "merokok": ["smoking_status"],
}
KLINIS = ["usia (age)", "hipertensi", "penyakit jantung", "glukosa"]
SOSIAL = ["jenis kelamin", "status menikah", "pekerjaan", "wilayah", "merokok"]


def kolom(grup):
    """Ubah nama kelompok jadi daftar kolom.

    Kolom kategorikal dicocokkan ke kolom one-hot: 'gender' cocok dengan 'gender_Male'.
    Turunan if-else ditambahkan apa adanya. dict.fromkeys membuang kolom kembar.
    """
    kol = []
    for g in grup:
        if g == "turunan if-else":
            kol += TURUNAN
        else:
            for k in GRUP[g]:
                kol += [c for c in X_dasar.columns if c == k or c.startswith(k + "_")]
    return list(dict.fromkeys(kol))


Delapan kombinasi yang diadu, dari yang paling ramping sampai paling lengkap. Jumlah
fiturnya adalah jumlah kolom setelah one-hot dan turunan ikut dihitung.


In [ ]:
KOMBINASI = {
    "A. Usia saja": ["usia (age)"],
    "B. Usia + glukosa": ["usia (age)", "glukosa"],
    "C. Usia + glukosa + hipertensi + bmi": ["usia (age)", "glukosa", "hipertensi", "bmi"],
    "D. Usia + hipertensi + penyakit jantung": ["usia (age)", "hipertensi", "penyakit jantung"],
    "E. Inti klinis": KLINIS,
    "F. Inti klinis + bmi": KLINIS + ["bmi"],
    "G. Semua fitur dasar": KLINIS + ["bmi"] + SOSIAL,
    "H. Semua fitur + turunan if-else": KLINIS + ["bmi"] + SOSIAL + ["turunan if-else"],
}

ringkas = pd.DataFrame(
    [{"Kombinasi": nama, "jumlah fitur": len(kolom(grup))}
     for nama, grup in KOMBINASI.items()]
)
ringkas


### 6b. Alat ukur, semuanya dibuat manual

Tidak ada scikit-learn di sini, termasuk untuk pembagian lipatan dan perhitungan metrik.

- **Lipatan stratified manual.** Tiap kelas diacak sendiri lalu dibagi rata ke lima
  lipatan, supaya proporsi stroke tiap lipatan tetap.
- **Standardisasi manual.** Jarak wajib memakai fitur berskala sama. Mean dan simpangan
  baku dihitung dari data latih lipatan saja, lalu dipakai ke data ujinya, supaya tidak
  ada kebocoran.
- **Metrik manual.** AP (tie-aware, sama seperti scikit-learn), precision pada recall
  0,80, ROC-AUC, dan selang kepercayaan bootstrap 95% untuk AP.


In [ ]:
def lipatan_stratified(y, n_lipatan=5, seed=42):
    """Bagi indeks jadi n lipatan dengan proporsi kelas tiap lipatan tetap."""
    acak = np.random.default_rng(seed)
    lipatan = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        acak.shuffle(idx)
        lipatan[idx] = np.arange(len(idx)) % n_lipatan
    return lipatan


def standardisasi(X_latih, X_uji):
    """Standardisasi memakai mean dan std data latih saja."""
    mu = X_latih.mean(axis=0)
    sd = X_latih.std(axis=0)
    sd[sd == 0] = 1.0
    return (X_latih - mu) / sd, (X_uji - mu) / sd


Skor confidence KNN adalah **proporsi tetangga yang stroke** dari 15 tetangga
terdekat. Jaraknya dihitung manual dengan numpy, tanpa scikit-learn.


In [ ]:
def knn_skor(X_latih, y_latih, X_uji, k=15):
    """Skor KNN: proporsi tetangga terdekat yang stroke. Jarak dihitung manual."""
    jarak2 = ((X_uji ** 2).sum(1)[:, None] + (X_latih ** 2).sum(1)[None, :]
              - 2 * X_uji @ X_latih.T)
    tetangga = np.argpartition(jarak2, k - 1, axis=1)[:, :k]
    return y_latih[tetangga].mean(axis=1)


def skor_oof(X_tab, y, k=15, n_lipatan=5, seed=42):
    """Skor out-of-fold KNN pada lipatan stratified."""
    lipatan = lipatan_stratified(y, n_lipatan, seed)
    skor = np.zeros(len(y))
    for f in range(n_lipatan):
        uji = lipatan == f
        X_latih, X_uji = standardisasi(X_tab[~uji], X_tab[uji])
        skor[uji] = knn_skor(X_latih, y[~uji], X_uji, k)
    return skor


Skor KNN bernilai diskret, jadi kurva precision-recall dihitung pada tiap nilai skor
unik. Cara ini sama dengan yang dipakai scikit-learn, sehingga angka AP-nya bisa diadu
langsung dengan notebook `13`.


In [ ]:
def kurva_pr(y_true, skor):
    """Presisi dan recall pada tiap nilai skor unik (tie-aware, sama seperti scikit-learn)."""
    urut = np.argsort(-skor, kind="mergesort")
    yt = y_true[urut].astype(float)
    nilai = skor[urut]
    batas = np.r_[np.where(np.diff(nilai) != 0)[0], len(nilai) - 1]
    tp = np.cumsum(yt)[batas]
    fp = 1 + batas - tp
    if tp[-1] == 0:
        return np.array([0.0]), np.array([0.0])
    return tp / (tp + fp), tp / tp[-1]


def average_precision(y_true, skor):
    """Luas di bawah kurva precision-recall."""
    pres, rec = kurva_pr(y_true, skor)
    pres = np.r_[pres[::-1], 1.0]
    rec = np.r_[rec[::-1], 0.0]
    return float(-np.sum(np.diff(rec) * pres[:-1]))


def presisi_pada_recall(y_true, skor, target=0.80):
    """Precision terbesar yang masih mencapai recall >= target."""
    pres, rec = kurva_pr(y_true, skor)
    idx = np.where(rec >= target)[0]
    return float(pres[idx].max()) if len(idx) else float("nan")


def roc_auc(y_true, skor):
    """ROC-AUC manual lewat peringkat (statistik Mann-Whitney)."""
    urut = np.argsort(skor, kind="mergesort")
    peringkat = np.empty(len(skor), dtype=float)
    peringkat[urut] = np.arange(1, len(skor) + 1)
    nilai = skor[urut]
    i = 0
    while i < len(nilai):
        j = i
        while j + 1 < len(nilai) and nilai[j + 1] == nilai[i]:
            j += 1
        if j > i:
            peringkat[urut[i:j + 1]] = (i + j + 2) / 2
        i = j + 1
    pos = y_true == 1
    n1, n0 = pos.sum(), (~pos).sum()
    return float((peringkat[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def bootstrap_ap(y_true, skor, B=2000, seed=42):
    """Selang kepercayaan 95% untuk average precision."""
    acak = np.random.default_rng(seed)
    n = len(y_true)
    nilai = []
    for _ in range(B):
        idx = acak.integers(0, n, n)
        if y_true[idx].sum() == 0:
            continue
        nilai.append(average_precision(y_true[idx], skor[idx]))
    return float(np.percentile(nilai, 2.5)), float(np.percentile(nilai, 97.5))


### 6c. Hasil: confidence level tiap kombinasi

Semua kombinasi diukur dengan model dan protokol yang sama. Diurutkan dari AP terbesar.


In [ ]:
baris = []
for nama, grup in KOMBINASI.items():
    kol = kolom(grup)
    skor = skor_oof(X_komb[kol].to_numpy(), y_komb)
    ap = average_precision(y_komb, skor)
    lo, hi = bootstrap_ap(y_komb, skor)
    baris.append({
        "Kombinasi atribut": nama,
        "jumlah fitur": len(kol),
        "AP": round(ap, 4),
        "AP 2,5%": round(lo, 4),
        "AP 97,5%": round(hi, 4),
        "precision @ recall 0,80": round(presisi_pada_recall(y_komb, skor), 4),
        "ROC-AUC": round(roc_auc(y_komb, skor), 4),
    })

tabel_knn = pd.DataFrame(baris).sort_values("AP", ascending=False).reset_index(drop=True)
tabel_knn


Diperlihatkan sebagai gambar. Batang abu-abu tipis adalah selang kepercayaan 95% dari
AP, dan itu bagian terpentingnya: hampir semua selang bertindihan.


In [ ]:
urut = tabel_knn.sort_values("AP")
galat = np.array([urut["AP"] - urut["AP 2,5%"], urut["AP 97,5%"] - urut["AP"]])

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(urut["Kombinasi atribut"], urut["AP"], xerr=galat, color="#4C78A8",
        error_kw={"ecolor": "#555", "capsize": 3})
ax.set_xlabel("Average precision (confidence level)")
ax.set_title('KNN manual: confidence terbaik di usia + glukosa, selangnya bertindihan')
plt.tight_layout()
plt.show()


**Yang terbaca dari tabel.**

1. Kombinasi dengan AP terbaik adalah **usia + glukosa** (AP 0,1443), disusul inti
   klinis + bmi (0,1333) dan usia + glukosa + hipertensi + bmi (0,1322).
2. Usia saja sudah membawa AP 0,1122. Menambah semua fitur dasar justru menurunkan AP
   ke 0,1052, dan menambah turunan if-else makin turun ke 0,1048. Jadi pada KNN,
   menambah atribut **tidak** menambah confidence.
3. Selang kepercayaan semua kombinasi bertindihan lebar, jadi selisih sekitar 0,01
   **tidak boleh** diklaim bermakna. Yang boleh diklaim hanya bedanya dengan kombinasi
   yang jauh lebih ramping.
4. Confidence KNN rendah: AP tertingginya 0,1443, sementara Logistic Regression di
   notebook `13` mencapai 0,184 dan Naive Bayes manual di `03b_naive_bayes_manual.ipynb`
   juga lebih tinggi. Ini sejalan dengan kesimpulan bagian 6 bahwa KNN paling lemah pada
   data ini.
5. Penyebabnya terlihat dari bentuk skornya: proporsi 15 tetangga hanya punya 16 nilai,
   jadi confidence-nya kasar dan model tetap cenderung menebak kelas mayoritas.


## 7. Kesimpulan

KNN manual berhasil dijalankan dan dievaluasi, sepenuhnya tanpa scikit-learn. Hasilnya
justru memperlihatkan kelemahan KNN pada data ini dengan sangat jelas.

| K | Akurasi | Presisi | Recall | F1 |
|---|---|---|---|---|
| 1 | 0,9197 | 0,2308 | 0,1475 | 0,1800 |
| 3 | 0,9285 | 0,1250 | 0,0328 | 0,0519 |
| 5 | 0,9285 | 0,0000 | 0,0000 | 0,0000 |
| 21 | 0,9403 | 0,0000 | 0,0000 | 0,0000 |

Pada k=5 dan seterusnya, dari 61 pasien stroke di data uji tidak satu pun terdeteksi
(TP=0, FN=61). Akurasi 0,93 itu menyesatkan: model hanya menebak "tidak stroke".

### Dibandingkan aturan medis

Skor CHA2DS2-VASc di bagian 5 tidak dilatih sama sekali, hanya penjumlahan poin, tetapi
hasilnya jauh lebih berguna:

| Cara | Recall | Presisi | F1 |
|---|---|---|---|
| Skor medis, ambang skor >= 2 | 0,6885 | 0,1368 | 0,2283 |
| Skor medis, ambang skor >= 3 | 0,5082 | 0,2109 | 0,2981 |
| KNN terbaik, yaitu k=1 | 0,1475 | 0,2308 | 0,1800 |
| KNN k=5 | 0,0000 | 0,0000 | 0,0000 |

Pada data ini **aturan sederhana dari buku kedokteran mengalahkan KNN**. Penyebabnya
sudah terlihat di atas: KNN menghitung jarak pada fitur yang belum diskalakan, sehingga
keputusannya kacau.

Beberapa catatan penting:

1. **Skala fitur sangat berpengaruh.** KNN menghitung jarak, jadi `avg_glucose_level`
   (rentang puluhan sampai ratusan) mendominasi `hypertension` yang hanya 0/1. Jarak
   antar pasien nyaris ditentukan glukosa saja. Versi scikit-learn nanti perlu scaling,
   dan itu akan mengubah hasil.
2. **Recall kelas stroke hampir nol**, bukan karena datanya tidak punya sinyal,
   melainkan karena jarak yang timpang plus kelas yang sangat tidak seimbang membuat
   voting didominasi tetangga "tidak stroke".
3. **K kecil menyelamatkan sedikit recall** (k=1 mendeteksi 9 pasien, k=3 hanya 2),
   tetapi F1 terbaiknya pun cuma 0,18 pada k=1.
4. **Aturan medis jadi pembanding yang jujur.** Karena skor CHA2DS2-VASc tidak belajar
   dari data, ia tidak bisa dituduh menghafal. Kalau model tidak mengalahkannya, berarti
   modelnya belum layak dipakai.

### Dibandingkan uji kombinasi atribut

Uji kombinasi di bagian 6 memperkuat kesimpulan dari jalur berbeda. KNN paling suka
kombinasi ramping (usia + glukosa), dan menambah fitur dasar maupun fitur turunan if-else
malah menurunkan confidence. Selang kepercayaannya lebar, jadi kesimpulannya hanya "tidak
ada bukti fitur tambahan menolong", bukan "membuangnya pasti lebih baik".

Langkah berikutnya: implementasi manual yang sama untuk Naive Bayes di
`03b_naive_bayes_manual.ipynb`, lalu versi scikit-learn di `04_model_library.ipynb`.